# Sales_Performance_Analysis

In [ ]:
import pandas as pd 
import matplotlib.pyplot as mpl 

In [2]:
customers = pd.read_csv("olist_customers_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")

### Check Missing Values

In [ ]:
customers.isnull().sum()
orders.isnull().sum()   
order_items.isnull().sum()
payments.isnull().sum()
products.isnull().sum()

### Check Duplicates

In [18]:
customers.duplicated().sum()
orders.duplicated().sum()
order_items.duplicated().sum()
payments.duplicated().sum()
products.duplicated().sum()

np.int64(0)

### Check Data Types

In [ ]:
#customers.info()
# orders.info()
# order_items.info()
# payments.info()
products.info()

### Data Cleaning

In [ ]:
# Cleaning the customers dataset by selecting only the relevant columns for our analysis
customers_clean = customers[
    ["customer_id", "customer_city", "customer_state"]
].copy()
customers_clean.head()

In [ ]:
# Cleaning the orders dataset by selecting only the relevant columns for our analysis
orders_clean = orders[
    [
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "order_status"
    ]
].copy()

orders_clean["order_purchase_timestamp"] = pd.to_datetime(
    orders_clean["order_purchase_timestamp"]
)
orders_clean.info()

In [29]:
# Cleaning the order_items dataset by selecting only the relevant columns for our analysis
order_items_clean = order_items[
    [
        "order_id",
        "product_id",
        "price",
        "freight_value"
    ]
].copy()

In [30]:
# Cleaning the payments dataset by selecting only the relevant columns for our analysis
payments_clean = payments[
    [
        "order_id",
        "payment_type",
        "payment_value"
    ]
].copy()

In [31]:
# Cleaning the products dataset by selecting only the relevant columns for our analysis
products_clean = products[
    [
        "product_id",
        "product_category_name"
    ]
].copy()

products_clean["product_category_name"] = (
    products_clean["product_category_name"]
    .fillna("Unknown")
)

### Validation Check

In [ ]:
customers_clean.info()
orders_clean.info()
order_items_clean.info()
payments_clean.info()
products_clean.info()

### Export Clean Files

In [33]:
customers_clean.to_csv(
    "cleaned_customers.csv",
    index=False
)

orders_clean.to_csv(
    "cleaned_orders.csv",
    index=False
)

order_items_clean.to_csv(
    "cleaned_order_items.csv",
    index=False
)

payments_clean.to_csv(
    "cleaned_payments.csv",
    index=False
)

products_clean.to_csv(
    "cleaned_products.csv",
    index=False
)

### Create Fact Sales Table

In [34]:
# merging customers and orders datasets to create a fact_sales dataset for analysis
fact_sales = orders_clean.merge(
    customers_clean,
    on="customer_id",
    how="left"
)

In [35]:
#merging fact_sales with order_items dataset to include product details in our analysis
fact_sales = fact_sales.merge(
    order_items_clean,
    on="order_id",
    how="left"
)

In [36]:
# merging fact_sales with products dataset to include product category details in our analysis
fact_sales = fact_sales.merge(
    products_clean,
    on="product_id",
    how="left"
)

In [37]:
# merging fact_sales with payments dataset to include payment details in our analysis
fact_sales = fact_sales.merge(
    payments_clean,
    on="order_id",
    how="left"
)

### Create Business Columns

In [40]:
fact_sales["sales"] = fact_sales["payment_value"] # renaming payment_value
fact_sales["freight_cost"] = fact_sales["freight_value"] # renaming freight_value
fact_sales["estimated_profit"] = (
    fact_sales["sales"]
    - fact_sales["freight_cost"]
) # calculating estimated profit by subtracting freight cost from sales

### Date Features

In [43]:
fact_sales["month"] = (
    fact_sales["order_purchase_timestamp"]
    .dt.month_name()
)# extracting month name from order_purchase_timestamp for monthly analysis

fact_sales["year"] = (
    fact_sales["order_purchase_timestamp"]
    .dt.year
)# extracting year from order_purchase_timestamp for yearly analysis

fact_sales["quarter"] = (
    fact_sales["order_purchase_timestamp"]
    .dt.quarter
)# extracting quarter from order_purchase_timestamp for quarterly analysis

### Final Columns

In [44]:
fact_sales = fact_sales[
[
"order_id",
"order_purchase_timestamp",
"year",
"quarter",
"month",
"customer_city",
"customer_state",
"product_category_name",
"payment_type",
"sales",
"freight_cost",
"estimated_profit"
]
]

### Final Validation

In [ ]:
fact_sales.info()

fact_sales.isnull().sum()

fact_sales.duplicated().sum()

In [51]:
fact_sales["product_category_name"] = (
    fact_sales["product_category_name"]
    .fillna("Unknown")
) # filling missing product category names with "Unknown"

fact_sales["payment_type"] = (
    fact_sales["payment_type"]
    .fillna("Unknown")
) # filling missing payment types with "Unknown"

fact_sales = fact_sales.dropna(
    subset=["sales"]
) # dropping rows where sales is null as they are essential for our analysis

fact_sales["freight_cost"] = (
    fact_sales["freight_cost"]
    .fillna(0)
)# filling missing freight costs with 0 as it indicates no freight cost for those orders

In [52]:
fact_sales["estimated_profit"] = (
    fact_sales["sales"]
    - fact_sales["freight_cost"]
)# recalculating estimated profit after filling missing freight costs

In [54]:
fact_sales.duplicated().sum()

np.int64(12643)

In [56]:
fact_sales[
    fact_sales.duplicated(keep=False)
].head(20)
fact_sales[
    fact_sales.duplicated()
].shape

(12643, 12)

In [ ]:
duplicate_rows = fact_sales[fact_sales.duplicated()]
duplicate_rows.head()

In [64]:
fact_sales = fact_sales.drop_duplicates()

In [65]:
fact_sales.shape
fact_sales.duplicated().sum()

np.int64(0)

In [ ]:
fact_sales.info()

fact_sales.isnull().sum()

fact_sales.duplicated().sum()

In [70]:
fact_sales.to_csv("fact_sales.csv", index=False)